In [1]:
from typing import TypedDict

In [2]:
class Person(TypedDict):
    name: str 
    age: int 

In [3]:
new_person: Person = {
    'name': "tipto",
    'age': 22
}
print(new_person)

{'name': 'tipto', 'age': 22}


In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [9]:
# define the model
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    task = "text-generation",
    temperature = 0.1,
    max_new_tokens = 512
)
model = ChatHuggingFace(llm = llm)

In [10]:
# schema
class Review(TypedDict):
    summary: str 
    sentiment: str 
    reviewed_by: str 

In [11]:
structured_model = model.with_structured_output(schema = Review)

result = structured_model.invoke(
    """I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, 
    it's an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything
    lightning fast—whether I'm gaming, multitasking, or editing photos. 
    The 5000mAh battery easily lasts a full day even with heavy use, and the 45W 
    fast charging is a lifesaver.

    The S-Pen integration is a great touch for note-taking and quick sketches, 
    though I don't use it often. What really blew me away is the 200MP camera—the
    night mode is stunning, capturing crisp, vibrant images even in low light.
    Zooming up to 100x actually works well for distant objects, 
    but anything beyond 30x loses quality.

    However, the weight and size make it a bit uncomfortable for one-handed use. 
    Also, Samsung’s One UI still comes with bloatware—why do I need five different 
    Samsung apps for things Google already provides? The $1,300 price tag is 
    also a hard pill to swallow.

    Pros:
    Insanely powerful processor (great for gaming and productivity)
    Stunning 200MP camera with incredible zoom capabilities
    Long battery life with fast charging
    S-Pen support is unique and useful
    
    Review by Tipto Ghosh
""")
result

{'summary': "The Samsung Galaxy S24 Ultra is an impressive device with a powerful processor, long battery life, and a stunning camera. However, it's quite heavy and expensive, and comes with unnecessary bloatware.",
 'sentiment': 'Positive with some reservations',
 'reviewed_by': 'Tipto Ghosh'}

- Sometimes it might happens that by just providing a single keyword like `summary` our 
model dont undertand that what to do then we can use `annotated` where we can provide a 
brief of what actually need to do.

In [17]:
from typing import Annotated, Optional, Literal

In [18]:
# schema
class Review(TypedDict):
    summary: Annotated[str , "A brief summary of the review"] 
    sentiment: Annotated[Literal["+ve" , "-ve"] , "Return the sentiment of the review either negative , positive or neutral"] 
    reviewed_by: str 
    key_things: Annotated[list[str] , "Write down all the key themes discussed in the review list"]
    pros: Annotated[Optional[list[str]] , "Write down all the pros inside a list"]
    cons: Annotated[Optional[list[str]] , "Write down all the cons inside a list"]

In [19]:
structured_model = model.with_structured_output(schema = Review)

result = structured_model.invoke(
    """I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, 
    it's an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything
    lightning fast—whether I'm gaming, multitasking, or editing photos. 
    The 5000mAh battery easily lasts a full day even with heavy use, and the 45W 
    fast charging is a lifesaver.

    The S-Pen integration is a great touch for note-taking and quick sketches, 
    though I don't use it often. What really blew me away is the 200MP camera—the
    night mode is stunning, capturing crisp, vibrant images even in low light.
    Zooming up to 100x actually works well for distant objects, 
    but anything beyond 30x loses quality.

    However, the weight and size make it a bit uncomfortable for one-handed use. 
    Also, Samsung’s One UI still comes with bloatware—why do I need five different 
    Samsung apps for things Google already provides? The $1,300 price tag is 
    also a hard pill to swallow.

    Pros:
    Insanely powerful processor (great for gaming and productivity)
    Stunning 200MP camera with incredible zoom capabilities
    Long battery life with fast charging
    S-Pen support is unique and useful
    
    Review by Tipto Ghosh
""")
result

{'summary': "The Samsung Galaxy S24 Ultra is a powerful device with a strong processor, impressive camera, and long battery life. However, it's heavy and expensive, and comes with unnecessary bloatware.",
 'sentiment': '+ve',
 'reviewed_by': 'Tipto Ghosh',
 'key_things': ['Powerful processor',
  'Stunning 200MP camera',
  'Long battery life',
  'S-Pen support',
  'Heavy and uncomfortable for one-handed use',
  'Bloatware',
  'High price'],
 'pros': ['Insanely powerful processor (great for gaming and productivity)',
  'Stunning 200MP camera with incredible zoom capabilities',
  'Long battery life with fast charging',
  'S-Pen support is unique and useful'],
 'cons': ['Heavy and uncomfortable for one-handed use',
  'Samsung’s One UI comes with bloatware',
  'High price tag']}

In [21]:
from pydantic import BaseModel 

class Student(BaseModel):
    name: str 

In [22]:
new_student = {
    'name': 'tipto'
}
student = Student(**new_student)
print(student)

name='tipto'


In [23]:
type(student)

__main__.Student

In [24]:
new_student = {
    'name': 32
}
student = Student(**new_student)
print(student)

ValidationError: 1 validation error for Student
name
  Input should be a valid string [type=string_type, input_value=32, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

### Setting up default value in pydantice

In [30]:
class Student(BaseModel):
    name: str = "no name given"
    age: Optional[int] = None

In [31]:
s1 = Student(*{})

In [32]:
s1.name

'no name given'

In [33]:
print(s1)

name='no name given' age=None


In [34]:
s2 = Student(**{
    'name': 'tipto',
    'age': '32' # pydantice can do coerce[convert data types]
})
type(s2.age)

int

In [35]:
from pydantic import Field
# applying constraints
class Student(BaseModel):
    name: str = "no name given"
    age: Optional[int] = None
    cgpa: float = Field(gt = 0 , lt = 4.00)

In [37]:
s3 = Student(**{
    'name': 'tipto',
    'age': 22,
    'cgpa': 3.99
})
s3

Student(name='tipto', age=22, cgpa=3.99)

## Using Pydantic with model

In [38]:
model

ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id='Qwen/Qwen2.5-7B-Instruct', temperature=0.1, stop_sequences=[], server_kwargs={}, model_kwargs={}, model='Qwen/Qwen2.5-7B-Instruct', client=<InferenceClient(model='Qwen/Qwen2.5-7B-Instruct', timeout=120)>, async_client=<InferenceClient(model='Qwen/Qwen2.5-7B-Instruct', timeout=120)>, task='text-generation'), model_id='Qwen/Qwen2.5-7B-Instruct', temperature=0.1, top_p=0.95, max_tokens=512, model_kwargs={})

In [39]:
# schema
class Review(BaseModel):
    summary: str = Field(
        description = "A brief summary of the review"
    ) 
    sentiment: Literal["+ve" , "-ve"] = Field(
      description = "Return the sentiment of the review either negative , positive or neutral"
    )  
    reviewed_by: str = Field(
        description = "name of the person who prvided the review",
        default = "No Name Given"
    ) 
    key_things: list[str] = Field(
        description = "Write down all the key themes discussed in the review list",
        default = [] # empty of no key things
    )
    pros: Optional[list[str]] = Field(
        description = "Write down all the pros inside a list" , default = []
    )
    cons: Optional[list[str]] = Field(
        description = "Write down all the cons inside a list", default = []
    )

In [43]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

In [42]:
# define the model
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    task = "text-generation",
    temperature = 0.1,
    max_new_tokens = 512
)
model = ChatHuggingFace(llm = llm)

In [44]:
# 1. Build a prompt that instructs the model to return JSON
prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract structured information from the review and return ONLY a valid JSON object with these exact fields:
    {{
        "summary": "brief summary of the review",
        "sentiment": "+ve or -ve",
        "reviewed_by": "name or 'No Name Given'",
        "key_things": ["list", "of", "key", "themes"],
        "pros": ["list of pros"],
        "cons": ["list of cons"]
    }}
    Return ONLY the JSON. No explanation, no markdown, no code blocks."""),
    ("human", "{review_text}")
])

In [49]:
def parse_review(review_text: str) -> Review:
    # 1. format the prompt
    messages = prompt.format_messages(review_text = review_text)
    # get the model's output
    output = model.invoke(messages)
    # get the content from output
    text = output.content.strip()
    # Strip markdown fences if model adds them anyway
    clean = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    
    data = json.loads(clean)
    return Review(**data)

In [50]:
review_text = """I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, 
    it's an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything
    lightning fast—whether I'm gaming, multitasking, or editing photos. 
    The 5000mAh battery easily lasts a full day even with heavy use, and the 45W 
    fast charging is a lifesaver.

    The S-Pen integration is a great touch for note-taking and quick sketches, 
    though I don't use it often. What really blew me away is the 200MP camera—the
    night mode is stunning, capturing crisp, vibrant images even in low light.
    Zooming up to 100x actually works well for distant objects, 
    but anything beyond 30x loses quality.

    However, the weight and size make it a bit uncomfortable for one-handed use. 
    Also, Samsung’s One UI still comes with bloatware—why do I need five different 
    Samsung apps for things Google already provides? The $1,300 price tag is 
    also a hard pill to swallow.

    Pros:
    Insanely powerful processor (great for gaming and productivity)
    Stunning 200MP camera with incredible zoom capabilities
    Long battery life with fast charging
    S-Pen support is unique and useful
    
    Review by Tipto Ghosh
"""

In [51]:
result = parse_review(review_text)
result

Review(summary="The Samsung Galaxy S24 Ultra is a powerful device with a great camera and long battery life, but it's heavy and expensive.", sentiment='+ve', reviewed_by='Tipto Ghosh', key_things=['powerful processor', 'stunning camera', 'long battery life', 'S-Pen support', 'weight and size'], pros=['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], cons=['Heavy and uncomfortable for one-handed use', 'Samsung’s One UI comes with bloatware', 'Expensive at $1,300'])

In [53]:
result.key_things

['powerful processor',
 'stunning camera',
 'long battery life',
 'S-Pen support',
 'weight and size']

In [54]:
result.cons

['Heavy and uncomfortable for one-handed use',
 'Samsung’s One UI comes with bloatware',
 'Expensive at $1,300']